# Level 2 — Skin wound sample-level export (per replicate)

Exports the **per-sample** (per-replicate) pseudobulk values that underlie the group-level DGE, so figures can show **mean ± SEM** across biological replicates.

Reproduces exactly the pseudobulk recipe in `2_skin_comparision.ipynb` (raw counts, QC filter `min_counts=30`/`min_genes=10`, sum per `sample_id`, gene prefilter total≥10, edgeR-style `log2(CPM+1)` with `prior_count=1`).

> **Note:** the dataset has no recorded animal IDs. The 17 samples are the KMeans-separated spatial sections; `mouse_id` is written as a PROVISIONAL label and must be replaced with real animal IDs before publishing SEM statistics.

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import warnings; warnings.filterwarnings('ignore')

COND_COL, SAMPLE_COL = 'condition', 'sample_id'
MIN_TOTAL_COUNTS, PRIOR_COUNT, SCALE = 10, 1.0, 1e6
OUT = '../../results/sample-level'
os.makedirs(OUT, exist_ok=True)

/Users/christoffer/miniconda3/envs/sc_py312/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Load + reproduce level-2 preprocessing (raw counts, same QC)

In [2]:
ad = sc.read_h5ad('../../data/talbot_xenium.h5ad')
ad = ad[ad.obs.tissue == 'skin'].copy()
ad.layers['raw'] = ad.X.copy()
sc.pp.calculate_qc_metrics(ad, percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(ad, min_counts=30)
sc.pp.filter_cells(ad, min_genes=10)
print(ad.n_obs, 'cells x', ad.n_vars, 'genes')

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


77401 cells x 5006 genes


## Pseudobulk: sum raw counts per sample_id → log2(CPM+1)

In [3]:
X = ad.layers['raw']
X = X.tocsr() if sparse.issparse(X) else np.asarray(X)
genes = pd.Index(ad.var_names)
samples, inv = np.unique(ad.obs[SAMPLE_COL].astype(str).values, return_inverse=True)
pb = np.zeros((X.shape[1], len(samples)), dtype=np.float64)
for g in range(len(samples)):
    idx = np.where(inv == g)[0]
    pb[:, g] = X[idx, :].sum(axis=0).A1 if sparse.issparse(X) else X[idx, :].sum(axis=0)
pb_counts = pd.DataFrame(pb, index=genes, columns=samples)
keep = pb_counts.sum(axis=1) >= MIN_TOTAL_COUNTS
pb_counts = pb_counts.loc[keep]
lib = pb_counts.sum(axis=0).values.reshape(1, -1)
cpm = (pb_counts.values * SCALE) / np.maximum(lib, 1.0)
logcpm_mat = pd.DataFrame(np.log2(cpm + PRIOR_COUNT), index=pb_counts.index, columns=pb_counts.columns)
logcpm_mat.index.name = 'gene'
print('logcpm_mat:', logcpm_mat.shape)

logcpm_mat: (4956, 17)


## Build sample_meta (with PROVISIONAL mouse_id) and save

In [4]:
obs = ad.obs
def per_sample(col):
    return obs.groupby(SAMPLE_COL)[col].agg(lambda s: s.astype(str).mode().iloc[0])
meta = pd.DataFrame({'sample_id': samples})
for c in ['condition','genotype','timepoint']:
    if c in obs.columns:
        meta[c] = meta['sample_id'].map(per_sample(c).to_dict())
if 'timepoint' not in meta.columns:
    meta['timepoint'] = meta['condition'].str.split('_').str[-1]
meta['n_cells'] = meta['sample_id'].map(obs[SAMPLE_COL].astype(str).value_counts().to_dict())
meta['total_pseudobulk_counts'] = meta['sample_id'].map(pb_counts.sum(axis=0).to_dict())
meta = meta.sort_values(['condition','sample_id']).reset_index(drop=True)
meta['replicate'] = meta.groupby('condition').cumcount() + 1
meta['mouse_id'] = meta['condition'] + '_m' + meta['replicate'].astype(str)
meta['mouse_id_status'] = 'PROVISIONAL_from_kmeans_section__replace_with_real_animal_ID'
meta = meta[['sample_id','mouse_id','mouse_id_status','condition','genotype','timepoint','replicate','n_cells','total_pseudobulk_counts']]
logcpm_mat = logcpm_mat[meta['sample_id'].tolist()]
logcpm_mat.to_csv(f'{OUT}/logcpm_mat.csv')
pb_counts[meta['sample_id'].tolist()].to_csv(f'{OUT}/pseudobulk_counts_mat.csv')
meta.to_csv(f'{OUT}/sample_meta.csv', index=False)
meta

,sample_id,mouse_id,mouse_id_status,condition,genotype,timepoint,replicate,n_cells,total_pseudobulk_counts
0,output-XETG00045__0059976__cre_24h__20250725__...,cre_24h_m1,PROVISIONAL_from_kmeans_section__replace_with_...,cre_24h,cre,24h,1,2610,203876.0
1,output-XETG00045__0059976__cre_24h__20250725__...,cre_24h_m2,PROVISIONAL_from_kmeans_section__replace_with_...,cre_24h,cre,24h,2,2173,182583.0
2,output-XETG00045__0059976__cre_24h__20250725__...,cre_24h_m3,PROVISIONAL_from_kmeans_section__replace_with_...,cre_24h,cre,24h,3,2908,229825.0
3,output-XETG00045__0059976__cre_72h__20250725__...,cre_72h_m1,PROVISIONAL_from_kmeans_section__replace_with_...,cre_72h,cre,72h,1,8550,753799.0
4,output-XETG00045__0059976__cre_72h__20250725__...,cre_72h_m2,PROVISIONAL_from_kmeans_section__replace_with_...,cre_72h,cre,72h,2,4456,479148.0
5,output-XETG00045__0059976__cre_72h__20250725__...,cre_72h_m3,PROVISIONAL_from_kmeans_section__replace_with_...,cre_72h,cre,72h,3,4116,400144.0
6,output-XETG00045__0059976__litt_24h__20250725_...,litt_24h_m1,PROVISIONAL_from_kmeans_section__replace_with_...,litt_24h,litt,24h,1,4727,361656.0
7,output-XETG00045__0059976__litt_24h__20250725_...,litt_24h_m2,PROVISIONAL_from_kmeans_section__replace_with_...,litt_24h,litt,24h,2,4105,425301.0
8,output-XETG00045__0059976__litt_24h__20250725_...,litt_24h_m3,PROVISIONAL_from_kmeans_section__replace_with_...,litt_24h,litt,24h,3,4619,372229.0
9,output-XETG00045__0059976__litt_72h__20250725_...,litt_72h_m1,PROVISIONAL_from_kmeans_section__replace_with_...,litt_72h,litt,72h,1,506,34917.0
